# Treinamento XGBoost — TriageAI

Classifica doenças a partir de sintomas usando **XGBoost + TF-IDF**.

> **Nota de modelagem:** O dataset Gold contém 773 classes (doenças). Com divisão treino/teste de 80/20,
> algumas classes raras aparecem apenas no conjunto de teste, o que invalida a avaliação.
> Por isso, filtramos para as **100 doenças mais frequentes**, que representam os casos de maior
> relevância clínica e garantem amostras suficientes em treino e teste para todas as classes.

- **Experimento MLflow:** `triageai-baseline-oficial`
- **Run name:** `xgboost_dataset_gold`
- **Dados:** `s3://gold/textos_rag.csv` (top-100 doenças)

In [1]:
import os
import pandas as pd
import mlflow
import mlflow.sklearn
import s3fs
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

print("Bibliotecas carregadas com sucesso.")

Bibliotecas carregadas com sucesso.


In [2]:
minio_storage_options = {
    "key": "minioadmin",
    "secret": "minioadmin123",
    "client_kwargs": {"endpoint_url": "http://minio:9000"}
}

tracking_uri = os.environ.get("MLFLOW_TRACKING_URI", "http://mlflow:5000")
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("triageai-baseline-oficial")
print(f"MLflow conectado em: {tracking_uri}")

MLflow conectado em: http://mlflow:5000


In [3]:
path_gold = "s3://gold/textos_rag.csv"
print(f"Lendo dados: {path_gold}")

df_full = pd.read_csv(path_gold, storage_options=minio_storage_options)
print(f"Dataset completo: {len(df_full)} linhas | {df_full['diseases'].nunique()} classes")

# Filtrar para as 100 doenças mais frequentes
TOP_N = 100
top_diseases = df_full['diseases'].value_counts().head(TOP_N).index.tolist()
df = df_full[df_full['diseases'].isin(top_diseases)].copy()
print(f"\nApós filtro top-{TOP_N}: {len(df)} linhas | {df['diseases'].nunique()} classes")

X = df['sintomas']
y = df['diseases']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Treino: {len(X_train)} | Teste: {len(X_test)}")

Lendo dados: s3://gold/textos_rag.csv
Dataset completo: 246945 linhas | 773 classes

Após filtro top-100: 101903 linhas | 100 classes
Treino: 81522 | Teste: 20381


In [4]:
# LabelEncoder garante labels inteiros 0..N-1 consistentes entre treino e teste
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)
num_classes = len(le.classes_)
print(f"Classes codificadas: {num_classes}")

Classes codificadas: 100


In [5]:
with mlflow.start_run(run_name="xgboost_dataset_gold"):

    n_estimators  = 100
    max_depth     = 6
    learning_rate = 0.1

    mlflow.log_param("model_type",      "XGBoost")
    mlflow.log_param("vectorizer",      "TF-IDF")
    mlflow.log_param("dataset_version", "Gold_v1")
    mlflow.log_param("dataset_path",    path_gold)
    mlflow.log_param("top_n_classes",   TOP_N)
    mlflow.log_param("num_classes",     num_classes)
    mlflow.log_param("n_estimators",    n_estimators)
    mlflow.log_param("max_depth",       max_depth)
    mlflow.log_param("learning_rate",   learning_rate)

    tfidf = TfidfVectorizer(max_features=10000)
    X_train_tfidf = tfidf.fit_transform(X_train)
    X_test_tfidf  = tfidf.transform(X_test)

    clf = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        objective="multi:softmax",
        num_class=num_classes,
        tree_method="hist",
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1,
        verbosity=0,
    )

    print("Treinando XGBoost...")
    clf.fit(X_train_tfidf, y_train_enc)

    print("Avaliando...")
    y_pred_enc = clf.predict(X_test_tfidf)

    acc       = accuracy_score(y_test_enc, y_pred_enc)
    f1        = f1_score(y_test_enc, y_pred_enc, average='weighted', zero_division=0)
    precision = precision_score(y_test_enc, y_pred_enc, average='weighted', zero_division=0)
    recall    = recall_score(y_test_enc, y_pred_enc, average='weighted', zero_division=0)

    mlflow.log_metric("accuracy",           acc)
    mlflow.log_metric("f1_score",           f1)
    mlflow.log_metric("precision_weighted", precision)
    mlflow.log_metric("recall_weighted",    recall)

    print(f"\nMétricas Finais (top-{TOP_N} doenças):")
    print(f"  Acurácia:         {acc:.4f}")
    print(f"  F1-Score (wt):    {f1:.4f}")
    print(f"  Precision (wt):   {precision:.4f}")
    print(f"  Recall (wt):      {recall:.4f}")

    pipeline = Pipeline([('tfidf', tfidf), ('clf', clf)])
    mlflow.sklearn.log_model(pipeline, "modelo_classificacao")
    print("\n✅ XGBoost treinado e registrado no MLflow com sucesso.")

Treinando XGBoost...
Avaliando...

Métricas Finais (top-100 doenças):
  Acurácia:         0.8747
  F1-Score (wt):    0.8752
  Precision (wt):   0.8767
  Recall (wt):      0.8747

✅ XGBoost treinado e registrado no MLflow com sucesso.
